# Predicción de Incumplimiento de SLAs en Tickets de Soporte
Este notebook implementa un flujo completo de Machine Learning para predecir el incumplimiento de SLAs.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import shap
import joblib

## 1. Generación de Datos Simulados

In [ ]:
# Generación de dataset de 5000 registros
np.random.seed(42)
n_samples = 5000

categorias = ['Redes', 'Hardware', 'Software', 'BD']
prioridades = ['Baja', 'Media', 'Alta']
seniorities = ['Junior', 'Semi-Senior', 'Senior']

# Variables categóricas
categoria = np.random.choice(categorias, n_samples)
prioridad = np.random.choice(prioridades, n_samples)
seniority = np.random.choice(seniorities, n_samples)
hora_creacion = np.random.randint(8, 19, n_samples) # 8 a 18 hrs

# Variable objetivo con desbalance (15% de incumplimiento)
incumplimiento_sla = np.random.choice([0, 1], size=n_samples, p=[0.85, 0.15])

df = pd.DataFrame({
    'Categoria': categoria,
    'Prioridad': prioridad,
    'Seniority': seniority,
    'Hora_Creacion': hora_creacion,
    'Incumplimiento_SLA': incumplimiento_sla
})
display(df.head())

## 2. Análisis Exploratorio de Datos (EDA)

In [ ]:
plt.figure(figsize=(14, 5))

# Gráfico 1: Desbalance de clases
plt.subplot(1, 2, 1)
sns.countplot(data=df, x='Incumplimiento_SLA', palette='viridis')
plt.title('Distribución de Incumplimiento de SLA (Desbalance)')
plt.xlabel('Incumplimiento SLA')
plt.ylabel('Cantidad')

# Gráfico 2: Tasa de incumplimiento por hora de creación
plt.subplot(1, 2, 2)
tasa_por_hora = df.groupby('Hora_Creacion')['Incumplimiento_SLA'].mean().reset_index()
sns.barplot(data=tasa_por_hora, x='Hora_Creacion', y='Incumplimiento_SLA', palette='magma')
plt.title('Tasa de Incumplimiento por Hora de Creación')
plt.xlabel('Hora de Creación')
plt.ylabel('Tasa de Incumplimiento')

plt.tight_layout()
plt.show()

## 3. Preprocesamiento de Datos

In [ ]:
# One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=['Categoria', 'Prioridad', 'Seniority'], drop_first=True)

X = df_encoded.drop('Incumplimiento_SLA', axis=1)
y = df_encoded['Incumplimiento_SLA']

# División: 70% Train, 30% Temp (Val + Test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
# División: 15% Validation, 15% Test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# Escalado
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Aplicar SMOTE solo en Train
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(f"Tamaño Train original: {X_train.shape}, Tras SMOTE: {X_train_smote.shape}")
print(f"Tamaño Val: {X_val.shape}")
print(f"Tamaño Test: {X_test.shape}")

## 4. Modelado
### 4.1 Random Forest

In [ ]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_smote, y_train_smote)

y_pred_rf = rf_model.predict(X_test_scaled)
print("Classification Report - Random Forest:\n")
print(classification_report(y_test, y_pred_rf))

### 4.2 XGBoost Classifier

In [ ]:
xgb_model = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_model.fit(X_train_smote, y_train_smote)

y_pred_xgb = xgb_model.predict(X_test_scaled)
print("Classification Report - XGBoost:\n")
print(classification_report(y_test, y_pred_xgb))

### 4.3 Red Neuronal Profunda (MLP)

In [ ]:
mlp_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_smote.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

mlp_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = mlp_model.fit(
    X_train_smote, y_train_smote,
    epochs=100,
    batch_size=32,
    validation_data=(X_val_scaled, y_val),
    callbacks=[early_stopping],
    verbose=0
)

# Predicciones con umbral 0.5
y_pred_mlp_prob = mlp_model.predict(X_test_scaled)
y_pred_mlp = (y_pred_mlp_prob > 0.5).astype(int)

print("\nClassification Report - MLP:\n")
print(classification_report(y_test, y_pred_mlp))

## 5. Explicabilidad con SHAP (XGBoost)

In [ ]:
# Usar un explainer para XGBoost
explainer = shap.TreeExplainer(xgb_model)
# SHAP values sobre el conjunto de test
shap_values = explainer.shap_values(X_test_scaled)

# Gráfico de resumen
shap.summary_plot(shap_values, X_test_scaled, feature_names=X.columns)

## 6. Exportación del Modelo

In [ ]:
joblib.dump(xgb_model, 'modelo_xgboost.pkl')
print("Modelo guardado como 'modelo_xgboost.pkl'")